# Getting Started
This tutorial demonstrates the configuration and use of a simple BSK-RL environment.
BSK-RL and dependencies should already be installed at this point (see [Installation](../install.rst)
if you haven't installed the package yet).

## Load Modules
In this tutorial, the environment will be created with `gym.make`, so it is necessary to
import the top-level `bsk_rl` module as well as `gym` and `bsk_rl` components.

In [ ]:
import numpy as np
from functools import partial
from bsk_rl import act, obs, sats, ConstellationTasking
from bsk_rl.sim import dyn, fsw
from bsk_rl.utils.orbital import relative_to_chief, random_orbit

from Basilisk.architecture import bskLogging

bskLogging.setDefaultLogLevel(bskLogging.BSK_WARNING)


If no errors were raised, you have a functional installation of `bsk_rl`.

## Configure the Satellite
[Satellites](../api_reference/sats/index.rst) are configurable agents in the environment.
To make a new environment, start by specifying the [observations](../api_reference/obs/index.rst)
and [actions](../api_reference/act/index.rst) of a satellite type, as well as the underlying
Basilisk [simulation](../api_reference/sim/index.rst) models used by the satellite.

In [ ]:
class TumbleSat(sats.Satellite):
    observation_spec = [
        obs.SatProperties(dict(prop="r_BN_N"), dict(prop="v_BN_N")),
    ]
    action_spec = [act.Drift()]
    dyn_type = dyn.ConjunctionDynModel
    fsw_type = fsw.BasicFSWModel


class ThrustSat(sats.Satellite):
    observation_spec = [
        obs.SatProperties(dict(prop="r_BN_N"), dict(prop="v_BN_N")),
        obs.RelativeProperties(
            dict(prop="r_DC_N"),
            chief_name="Tumbler",
        ),
    ]
    action_spec = [act.MagicThrust(max_dv=100)]
    dyn_type = dyn.ConjunctionDynModel
    fsw_type = fsw.MagicOrbitalManeuverFSWModel


## Making the Environment
For this example, we will be using the single-agent [SatelliteTasking](../api_reference/index.rst) 
environment. Along with passing the satellite that we configured, the environment takes
a [scenario](../api_reference/scene/index.rst), which defines the environment the
satellite is acting in, and a [rewarder](../api_reference/data/index.rst), which defines
how data collected from the scenario is rewarded.

In [ ]:
env = ConstellationTasking(
    satellites=[
        TumbleSat("Tumbler"),
        ThrustSat("Thrust-1"),
        ThrustSat("Thrust-2"),
    ],
    sat_arg_randomizer=relative_to_chief(
        chief_name="Tumbler",
        chief_orbit=partial(random_orbit, i=None),
        deputy_relative_state={
            "Thrust-1": np.array([50, 0, 0, -3, 0, 0]),
            "Thrust-2": np.array([-50, 0, 0, 3, 0, 0]),
        },
    ),
    time_limit=5700.0 * 3,
    log_level="INFO",
)
env.action_spaces

## Interacting with the Environment

First, the environment is reset.

In [ ]:
observation, info = env.reset(seed=0)
print(observation)
print(info)

Next, we take the scan action (`action=0`) a few times. This allows for the satellite to
settle its attitude in the nadir pointing mode to satisfy imaging conditions. Note that 
the logs show little or no data accumulated in the first two steps as it settles, but
achieves 60 reward (corresponding to 60 seconds of imaging) by the third step.

In [ ]:
for _ in range(20):
    actions = {}
    # for sat in env.satellites:
    #     if sat.requires_retasking:
    #         if isinstance(sat, TumbleSat):
    #             actions[sat.name] = 0
    #         else:
    #             actions[sat.name] = np.concatenate(
    #                 (np.random.uniform(-20, 20, 3), np.random.uniform(0, 200, 1))
    #             )

    observation, reward, terminated, truncated, info = env.step(actions)